# 🎯 Attention Mechanism From Scratch

> Understanding the heart of Transformers by implementing Self-Attention, Multi-Head Attention, and Positional Encoding using NumPy

**Topics covered:**
- Self-attention intuition
- Query, Key, Value (Q, K, V)
- Attention scores and softmax
- Multi-head attention
- Positional encoding
- Masked attention (for decoding)
- Complete transformer block

## 📚 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✅ Imports ready!")
print(f"NumPy version: {np.__version__}")

## 💡 Part 1: Intuition - What is Attention?

**Attention** allows the model to focus on relevant parts of the input when producing each output.

Think of it like reading a sentence: when you understand the word "it", your brain looks back at previous words to figure out what "it" refers to.

**Formula:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

## 🔧 Part 2: Self-Attention From Scratch

In [ ]:
class SelfAttention:
    """
    Self-Attention mechanism implemented from scratch
    
    Input: sequence of vectors (batch, seq_len, d_model)
    Output: attended vectors (batch, seq_len, d_model)
    """
    
    def __init__(self, d_model=64):
        """
        Args:
            d_model: Dimension of input vectors (embedding dimension)
        """
        self.d_model = d_model
        self.d_k = d_model  # Dimension of Q, K, V
        
        # Initialize weight matrices for Q, K, V projections
        # These are learned during training
        self.W_q = np.random.randn(d_model, self.d_k) * np.sqrt(2.0 / d_model)
        self.W_k = np.random.randn(d_model, self.d_k) * np.sqrt(2.0 / d_model)
        self.W_v = np.random.randn(d_model, self.d_k) * np.sqrt(2.0 / d_model)
        
        # Output projection
        self.W_o = np.random.randn(self.d_k, d_model) * np.sqrt(2.0 / self.d_k)
        
        # Store for visualization
        self.last_attention_weights = None
        self.last_Q = None
        self.last_K = None
        self.last_V = None
    
    def softmax(self, x, axis=-1):
        """Numerically stable softmax"""
        exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
        return exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    
    def forward(self, x, mask=None):
        """
        Forward pass of self-attention
        
        Args:
            x: Input of shape (batch_size, seq_len, d_model)
            mask: Optional mask for padded positions or causal masking
        
        Returns:
            output: Attended vectors (batch_size, seq_len, d_model)
            attention_weights: (batch_size, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape
        
        # Step 1: Project input to Q, K, V
        # Q = x @ W_q, K = x @ W_k, V = x @ W_v
        Q = np.dot(x, self.W_q)  # (batch, seq_len, d_k)
        K = np.dot(x, self.W_k)  # (batch, seq_len, d_k)
        V = np.dot(x, self.W_v)  # (batch, seq_len, d_k)
        
        self.last_Q = Q
        self.last_K = K
        self.last_V = V
        
        # Step 2: Compute attention scores
        # scores = Q @ K^T / sqrt(d_k)
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_k)
        # scores shape: (batch, seq_len, seq_len)
        
        # Step 3: Apply mask (if provided)
        if mask is not None:
            scores = scores + mask  # mask contains -inf for positions to ignore
        
        # Step 4: Apply softmax to get attention weights
        attention_weights = self.softmax(scores, axis=-1)
        self.last_attention_weights = attention_weights
        
        # Step 5: Apply attention weights to V
        # output = attention_weights @ V
        attended = np.matmul(attention_weights, V)  # (batch, seq_len, d_k)
        
        # Step 6: Final linear projection
        output = np.dot(attended, self.W_o)  # (batch, seq_len, d_model)
        
        return output, attention_weights
    
    def visualize_attention(self, tokens=None):
        """Visualize the attention weights"""
        if self.last_attention_weights is None:
            print("Run forward() first!")
            return
        
        # Take first sample from batch
        attn = self.last_attention_weights[0]  # (seq_len, seq_len)
        
        plt.figure(figsize=(8, 6))
        plt.imshow(attn, cmap='viridis', aspect='auto')
        plt.colorbar(label='Attention Weight')
        plt.title('Self-Attention Heatmap', fontsize=14, fontweight='bold')
        
        if tokens:
            plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
            plt.yticks(range(len(tokens)), tokens)
        
        plt.xlabel('Key Positions')
        plt.ylabel('Query Positions')
        plt.tight_layout()
        plt.show()

# Test self-attention
print("=== Testing Self-Attention ===")

# Create a simple input: 2 sequences, 4 tokens each, 8 dimensions
batch_size, seq_len, d_model = 2, 4, 8
x = np.random.randn(batch_size, seq_len, d_model)

# Create attention layer
attn = SelfAttention(d_model=d_model)

# Forward pass
output, weights = attn.forward(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nAttention weights (first sample):")
print(weights[0].round(3))
print("\n✅ Self-attention working!")

## 🧪 Part 3: Attention with Real Example

Let's see how attention works with a real sentence!

In [ ]:
# Simple word embeddings (normally these come from an embedding layer)
sentence = ["The", "cat", "sat", "on", "the", "mat"]

# Create simple embeddings (normally trained)
np.random.seed(42)
d_model = 16
embeddings = {}
for word in set(sentence):
    embeddings[word] = np.random.randn(d_model)

# Stack into sequence
seq_len = len(sentence)
x = np.array([embeddings[w] for w in sentence])  # (seq_len, d_model)
x = x.reshape(1, seq_len, d_model)  # Add batch dimension

print(f"Sentence: {' '.join(sentence)}")
print(f"Sequence shape: {x.shape}")

# Create attention and run
attn = SelfAttention(d_model=d_model)
output, weights = attn.forward(x)

# Visualize
attn.visualize_attention(tokens=sentence)

# Show which words each word attends to most
print("\n🔍 Attention Analysis:")
for i, word in enumerate(sentence):
    attn_scores = weights[0, i]
    top_idx = np.argmax(attn_scores)
    print(f"   '{word}' attends most to -> '{sentence[top_idx]}' (score: {attn_scores[top_idx]:.3f})")

## 🎭 Part 4: Multi-Head Attention

Instead of one attention, we use **multiple heads** that learn different types of relationships:
- Head 1: Subject-verb relationships
- Head 2: Pronoun references
- Head 3: Adjective-noun pairs
- etc.

In [ ]:
class MultiHeadAttention:
    """
    Multi-Head Attention: Run multiple attention heads in parallel
    """
    
    def __init__(self, d_model=64, num_heads=8):
        """
        Args:
            d_model: Total dimension of model
            num_heads: Number of parallel attention heads
        """
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Single weight matrices for all heads (more efficient)
        self.W_q = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        self.W_k = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        self.W_v = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        self.W_o = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        
        self.last_attention_weights = None
    
    def softmax(self, x, axis=-1):
        exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
        return exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, d_k)
        Input: (batch, seq_len, d_model)
        Output: (batch, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, _ = x.shape
        x = x.reshape(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)  # (batch, heads, seq_len, d_k)
    
    def combine_heads(self, x):
        """
        Reverse of split_heads
        Input: (batch, num_heads, seq_len, d_k)
        Output: (batch, seq_len, d_model)
        """
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)  # (batch, seq_len, heads, d_k)
        return x.reshape(batch_size, seq_len, self.d_model)
    
    def forward(self, x, mask=None):
        """
        Multi-head attention forward pass
        """
        batch_size, seq_len, _ = x.shape
        
        # Linear projections
        Q = np.dot(x, self.W_q)  # (batch, seq_len, d_model)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        # Split into heads
        Q = self.split_heads(Q)  # (batch, heads, seq_len, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Scaled dot-product attention for each head
        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        # scores: (batch, heads, seq_len, seq_len)
        
        if mask is not None:
            scores = scores + mask
        
        attention_weights = self.softmax(scores, axis=-1)
        self.last_attention_weights = attention_weights
        
        # Apply attention to values
        attended = np.matmul(attention_weights, V)
        # attended: (batch, heads, seq_len, d_k)
        
        # Combine heads
        attended = self.combine_heads(attended)
        
        # Final linear projection
        output = np.dot(attended, self.W_o)
        
        return output, attention_weights
    
    def visualize_heads(self, tokens=None):
        """Visualize attention for each head"""
        if self.last_attention_weights is None:
            print("Run forward() first!")
            return
        
        attn = self.last_attention_weights[0]  # (heads, seq_len, seq_len)
        
        n_heads = attn.shape[0]
        fig, axes = plt.subplots(2, n_heads//2, figsize=(14, 6))
        
        for h in range(n_heads):
            row, col = h // (n_heads//2), h % (n_heads//2)
            im = axes[row, col].imshow(attn[h], cmap='viridis', aspect='auto')
            axes[row, col].set_title(f'Head {h+1}', fontsize=10)
            
            if tokens:
                axes[row, col].set_xticks(range(len(tokens)))
                axes[row, col].set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
                axes[row, col].set_yticks(range(len(tokens)))
                axes[row, col].set_yticklabels(tokens, fontsize=8)
        
        plt.suptitle('Multi-Head Attention Visualization', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Test multi-head attention
print("=== Testing Multi-Head Attention ===")

d_model = 64
num_heads = 8
seq_len = 10
batch_size = 2

x = np.random.randn(batch_size, seq_len, d_model)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
output, weights = mha.forward(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"   (batch, heads, seq_len, seq_len) = {weights.shape}")
print("\n✅ Multi-head attention working!")

## 📍 Part 5: Positional Encoding

Transformers have no inherent sense of position (unlike RNNs). Positional encoding adds position information to embeddings.

**Formula (sinusoidal):**
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding:
    """
    Sinusoidal positional encoding
    """
    
    def __init__(self, d_model=64, max_len=5000):
        """
        Args:
            d_model: Dimension of embeddings
            max_len: Maximum sequence length to pre-compute
        """
        self.d_model = d_model
        
        # Create positional encoding matrix
        pe = np.zeros((max_len, d_model))
        
        # Position indices
        position = np.arange(0, max_len).reshape(-1, 1)
        
        # Divisor term
        div_term = np.exp(
            np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model)
        )
        
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        
        self.pe = pe
    
    def forward(self, x):
        """
        Add positional encoding to input
        
        Args:
            x: (batch_size, seq_len, d_model)
        """
        seq_len = x.shape[1]
        return x + self.pe[:seq_len]
    
    def visualize(self, seq_len=50):
        """Visualize positional encoding patterns"""
        plt.figure(figsize=(12, 6))
        plt.imshow(
            self.pe[:seq_len].T,
            cmap='RdBu_r',
            aspect='auto',
            interpolation='nearest'
        )
        plt.colorbar(label='Value')
        plt.xlabel('Position in Sequence')
        plt.ylabel('Embedding Dimension')
        plt.title('Positional Encoding Heatmap', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Show specific dimensions
        fig, axes = plt.subplots(1, 4, figsize=(14, 3))
        dims = [0, 1, 8, 9]
        for idx, dim in enumerate(dims):
            axes[idx].plot(self.pe[:seq_len, dim])
            axes[idx].set_title(f'Dimension {dim}')
            axes[idx].set_xlabel('Position')
            axes[idx].grid(True, alpha=0.3)
        
        plt.suptitle('Positional Encoding - Individual Dimensions', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Test positional encoding
print("=== Testing Positional Encoding ===")

pe = PositionalEncoding(d_model=64, max_len=100)

# Test with random embeddings
seq_len = 20
embeddings = np.random.randn(1, seq_len, 64)
encoded = pe.forward(embeddings)

print(f"Input shape: {embeddings.shape}")
print(f"Output shape: {encoded.shape}")
print(f"Positional encoding shape: {pe.pe.shape}")

# Visualize
pe.visualize(seq_len=50)

print("\n✅ Positional encoding working!")

## 🎭 Part 6: Masked Attention (Causal/Autoregressive)

For **decoding** (generating text), we mask future positions so the model can only attend to previous tokens.

In [ ]:
def create_causal_mask(seq_len):
    """
    Create a causal (look-ahead) mask
    
    Returns mask where positions > current position are masked
    """
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    mask = mask * -1e9  # Set to very negative number
    return mask

def create_padding_mask(seq_lens, max_len):
    """
    Create padding mask for variable length sequences
    
    Args:
        seq_lens: List of actual sequence lengths
        max_len: Maximum sequence length
    """
    mask = np.zeros((len(seq_lens), max_len, max_len))
    for i, length in enumerate(seq_lens):
        mask[i, :, length:] = -1e9
    return mask

# Demonstrate causal masking
print("=== Causal Masking ===")

seq_len = 5
causal_mask = create_causal_mask(seq_len)

print("Causal Mask (0 = attend, -inf = mask):")
print(causal_mask)

# Visualize
plt.figure(figsize=(6, 5))
plt.imshow(causal_mask == 0, cmap='RdYlGn', aspect='auto')
plt.colorbar(ticks=[0, 1], label='Can Attend')
plt.xticks(range(seq_len), [f'Pos {i}' for i in range(seq_len)])
plt.yticks(range(seq_len), [f'Pos {i}' for i in range(seq_len)])
plt.title('Causal Mask (Green = Can Attend)', fontsize=12, fontweight='bold')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.tight_layout()
plt.show()

# Test with attention
x = np.random.randn(1, seq_len, 16)
attn = SelfAttention(d_model=16)

# Without mask
out_no_mask, weights_no_mask = attn.forward(x)

# With causal mask
mask = causal_mask.reshape(1, seq_len, seq_len)
out_masked, weights_masked = attn.forward(x, mask=mask)

print("\nAttention weights WITHOUT mask (first position):")
print(weights_no_mask[0, 0].round(3))

print("\nAttention weights WITH causal mask (first position):")
print(weights_masked[0, 0].round(3))

print("\n✅ Causal masking working!")

## 🏗️ Part 7: Complete Transformer Block

In [ ]:
class LayerNorm:
    """Layer normalization"""
    
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
    
    def forward(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta


class FeedForward:
    """Position-wise feed-forward network"""
    
    def __init__(self, d_model=64, d_ff=256):
        self.W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def forward(self, x):
        hidden = self.relu(np.dot(x, self.W1) + self.b1)
        return np.dot(hidden, self.W2) + self.b2


class TransformerBlock:
    """
    Complete Transformer Encoder Block
    Multi-Head Attention + Add&Norm + FeedForward + Add&Norm
    """
    
    def __init__(self, d_model=64, num_heads=8, d_ff=256):
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)
        self.norm2 = LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        """
        Forward pass with residual connections
        """
        # Self-attention with residual
        attn_out, _ = self.attention.forward(x, mask)
        x = self.norm1(x + attn_out)  # Add & Norm
        
        # Feed-forward with residual
        ff_out = self.ff.forward(x)
        x = self.norm2(x + ff_out)  # Add & Norm
        
        return x

# Test transformer block
print("=== Testing Transformer Block ===")

d_model = 64
seq_len = 10
batch_size = 2

x = np.random.randn(batch_size, seq_len, d_model)

transformer = TransformerBlock(d_model=d_model, num_heads=8, d_ff=256)
output = transformer.forward(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output mean: {np.mean(output):.4f}, std: {np.std(output):.4f}")
print("\n✅ Transformer block working!")

## 🧪 Part 8: Full Example - Sequence Classification

In [ ]:
class SimpleTransformer:
    """
    Simple Transformer for sequence classification
    """
    
    def __init__(self, vocab_size=100, d_model=64, num_heads=8, num_layers=2, num_classes=2):
        self.d_model = d_model
        
        # Embedding layer
        self.embedding = np.random.randn(vocab_size, d_model) * 0.01
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model)
        
        # Transformer blocks
        self.blocks = [
            TransformerBlock(d_model, num_heads)
            for _ in range(num_layers)
        ]
        
        # Classification head
        self.classifier = np.random.randn(d_model, num_classes) * np.sqrt(2.0 / d_model)
        self.classifier_b = np.zeros(num_classes)
    
    def forward(self, x):
        """
        x: token indices (batch, seq_len)
        """
        # Embedding
        embedded = self.embedding[x]  # (batch, seq_len, d_model)
        
        # Add positional encoding
        x = self.pos_encoding.forward(embedded)
        
        # Pass through transformer blocks
        for block in self.blocks:
            x = block.forward(x)
        
        # Global average pooling
        x = np.mean(x, axis=1)  # (batch, d_model)
        
        # Classification
        logits = np.dot(x, self.classifier) + self.classifier_b
        
        return logits
    
    def predict(self, x):
        logits = self.forward(x)
        return np.argmax(logits, axis=1)

# Test simple transformer
print("=== Testing Simple Transformer ===")

vocab_size = 100
seq_len = 20
batch_size = 4

# Random token sequences
tokens = np.random.randint(0, vocab_size, size=(batch_size, seq_len))

model = SimpleTransformer(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=8,
    num_layers=2,
    num_classes=2
)

logits = model.forward(tokens)
predictions = model.predict(tokens)

print(f"Input tokens shape: {tokens.shape}")
print(f"Logits shape: {logits.shape}")
print(f"Predictions: {predictions}")
print(f"Prediction probabilities: {np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)}")
print("\n✅ Simple transformer working!")

## 📊 Part 9: Attention Pattern Analysis

In [ ]:
# Analyze what different attention heads learn
print("=== Attention Pattern Analysis ===")

# Create a meaningful sentence
sentence = ["The", "cat", "sat", "on", "the", "mat", "and", "looked", "at", "the", "bird"]
seq_len = len(sentence)
d_model = 64

# Create embeddings
np.random.seed(42)
embeddings = {w: np.random.randn(d_model) for w in set(sentence)}
x = np.array([embeddings[w] for w in sentence]).reshape(1, seq_len, d_model)

# Multi-head attention
mha = MultiHeadAttention(d_model=d_model, num_heads=4)
output, weights = mha.forward(x)

# Visualize all heads
mha.visualize_heads(tokens=sentence)

# Analyze each head
print("\n🔍 Head Analysis:")
for h in range(4):
    attn = weights[0, h]  # (seq_len, seq_len)
    
    # Find strongest attention for each position
    print(f"\nHead {h+1}:")
    for i in range(seq_len):
        top_j = np.argmax(attn[i])
        if attn[i, top_j] > 0.3:  # Only show strong attentions
            print(f"   '{sentence[i]}' -> '{sentence[top_j]}' (score: {attn[i, top_j]:.3f})")

print("\n✅ Analysis complete!")

## 🎯 Exercises

1. **Implement attention dropout**: Randomly zero out attention weights during training
2. **Add multiple transformer blocks**: Stack 2-3 blocks and compare performance
3. **Implement cross-attention**: Where Q comes from decoder, K/V from encoder
4. **Try different positional encodings**: Learned embeddings vs sinusoidal
5. **Visualize attention evolution**: Track how attention patterns change during training
6. **Implement beam search**: For text generation with the decoder

## 📚 Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **Self-Attention** | Q, K, V projections + softmax(QK^T/√d)V |
| **Multi-Head** | Multiple attention heads in parallel |
| **Positional Encoding** | Sinusoidal patterns add position info |
| **Causal Mask** | Prevents looking at future tokens |
| **Layer Norm** | Normalizes across features |
| **Residual Connections** | Helps gradients flow deeper |
| **Feed-Forward** | Position-wise MLP for transformation |

**Next:** Try `02-transformer-from-scratch.ipynb` for the complete GPT model! 🚀